<a href="https://colab.research.google.com/github/Cosmito-TheBeginning/ai-mental-health-app/blob/main/train_emotion_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets scikit_learn accelerate -q
print("Libraries Installed!")

Libraries Installed!


In [2]:
import re
import nltk
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

nltk.download('stopwords')
from nltk.corpus import stopwords
print(" All imports done!")

 All imports done!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
stop_words = set(stopwords.words('english'))
keep_words = {'not', 'no', 'never', 'nothing', "n't"}
stop_words = stop_words - keep_words

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join(text.split())
    return text

def remove_stopwords(text):
    words = text.split()
    return ' '.join([w for w in words if w not in stop_words])

sample = "I can't believe this!! http://t.co/xyz #sad @user"
print("Before:", sample)
print("After: ", remove_stopwords(clean_text(sample)))
print("✅ Cleaning functions ready!")

Before: I can't believe this!! http://t.co/xyz #sad @user
After:  cant believe
✅ Cleaning functions ready!


In [4]:
print("Loading GoEmotions dataset...")
print("⏳ Wait 1-2 minutes...")

dataset = load_dataset('go_emotions', 'simplified')

print(dataset)
print("✅ Dataset loaded!")

Loading GoEmotions dataset...
⏳ Wait 1-2 minutes...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
✅ Dataset loaded!


In [5]:
MODEL = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize_function(examples):
    cleaned = [remove_stopwords(clean_text(t)) for t in examples['text']]
    return tokenizer(
        cleaned,
        truncation=True,
        padding='max_length',
        max_length=128
    )

print("Tokenizing dataset...")
print("⏳ Wait 3-5 minutes...")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("✅ Stage 1 Complete!")
print(f"Train:      {len(tokenized_dataset['train'])} samples")
print(f"Validation: {len(tokenized_dataset['validation'])} samples")
print(f"Test:       {len(tokenized_dataset['test'])} samples")

Tokenizing dataset...
⏳ Wait 3-5 minutes...


Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

✅ Stage 1 Complete!
Train:      43410 samples
Validation: 5426 samples
Test:       5427 samples


In [6]:
def filter_single_label(example):
    return len(example['labels']) == 1

tokenized_dataset = tokenized_dataset.filter(filter_single_label)

def flatten_labels(example):
    example['labels'] = example['labels'][0]
    return example

tokenized_dataset = tokenized_dataset.map(flatten_labels)


tokenized_dataset.set_format(type='torch',columns=['input_ids', 'attention_mask', 'labels'])

print("✅ Dataset ready!")
print(f"Train:      {len(tokenized_dataset['train'])} samples")
print(f"Validation: {len(tokenized_dataset['validation'])} samples")
print(f"Test:       {len(tokenized_dataset['test'])} samples")

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

Map:   0%|          | 0/4548 [00:00<?, ? examples/s]

Map:   0%|          | 0/4590 [00:00<?, ? examples/s]

✅ Dataset ready!
Train:      36308 samples
Validation: 4548 samples
Test:       4590 samples


In [7]:
import torch
print("GPU available:", torch.cuda.is_available())

print("\nLoading RoBERTa model...")
print("⏳ Wait 2-3 minutes (downloading ~500MB)...")

model = AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=28)

print("✅ Model loaded!")
print(f"Total parameters: {model.num_parameters():,}")

GPU available: True

Loading RoBERTa model...
⏳ Wait 2-3 minutes (downloading ~500MB)...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded!
Total parameters: 124,667,164


In [8]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=100,
)

print("✅ Training config set!")
print(f"Epochs:        {training_args.num_train_epochs}")
print(f"Batch size:    {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

✅ Training config set!
Epochs:        10
Batch size:    16
Learning rate: 2e-05


In [9]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {
        'accuracy': round(accuracy, 4),
        'f1': round(f1, 4)
    }

print("✅ Metrics function ready!")
print("Measuring: Accuracy + F1 Score after each epoch")

✅ Metrics function ready!
Measuring: Accuracy + F1 Score after each epoch


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

print("✅ Trainer created!")
print(f"Train samples:      {len(tokenized_dataset['train'])}")
print(f"Validation samples: {len(tokenized_dataset['validation'])}")
print("\n Ready to train!")

✅ Trainer created!
Train samples:      36308
Validation samples: 4548

 Ready to train!


In [ ]:
print(" Training started!")
print(" ~15-20 minutes with GPU\n")

trainer.train()

print("\n TRAINING COMPLETE!")

 Training started!
 ~15-20 minutes with GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.529153,1.451368,0.600700,0.572600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]